# SENTINEL Modal Notebook

This notebook is the Modal-native training driver for SENTINEL.

Use it inside a Modal notebook to:
- clone or open the repo
- verify the hosted GPU path
- train the main agents
- evaluate trained agents
- download logs and checkpoints from the Modal volume
- generate reward and loss plots for judges

## Recommended order
1. `holmes`
2. `forge`
3. `hermes`
4. `oracle`
5. `argus`


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org-or-user>/sentinel.git"
BRANCH = "main"
REPO_DIR = Path("/root/sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(os.getcwd())


In [ ]:
!pip -q install modal pandas matplotlib

## Modal Auth

If your notebook image is already authenticated, you can skip this cell.

Otherwise, fill in your token values.

In [ ]:
import os
import sys
import subprocess

MODAL_TOKEN_ID = os.environ.get("MODAL_TOKEN_ID", "")
MODAL_TOKEN_SECRET = os.environ.get("MODAL_TOKEN_SECRET", "")

if MODAL_TOKEN_ID and MODAL_TOKEN_SECRET:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "modal",
            "token",
            "set",
            "--token-id",
            MODAL_TOKEN_ID,
            "--token-secret",
            MODAL_TOKEN_SECRET,
            "--profile",
            "default",
            "--activate",
        ],
        check=True,
    )
else:
    print("Skipping token set. Provide MODAL_TOKEN_ID and MODAL_TOKEN_SECRET env vars if needed.")

subprocess.run([sys.executable, "-m", "modal", "profile", "current"], check=True)


## Hosted GPU Sanity Check

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "modal", "run", "modal_train.py::gpu_sanity"], check=True)


## Train Core Agents

In [ ]:
import subprocess
import sys

TRAIN_PLAN = [
    {"agent": "holmes", "episodes": 300, "batch_size": 2},
    {"agent": "forge", "episodes": 300, "batch_size": 2},
]

for job in TRAIN_PLAN:
    cmd = [
        sys.executable,
        "-m",
        "modal",
        "run",
        "modal_train.py::train",
        "--agent",
        job["agent"],
        "--episodes",
        str(job["episodes"]),
        "--batch-size",
        str(job["batch_size"]),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


## Optional Supporting Agents

In [ ]:
import subprocess
import sys

RUN_OPTIONAL = False
OPTIONAL_PLAN = [
    {"agent": "hermes", "episodes": 200, "batch_size": 2},
    {"agent": "oracle", "episodes": 200, "batch_size": 2},
    {"agent": "argus", "episodes": 200, "batch_size": 2},
]

if RUN_OPTIONAL:
    for job in OPTIONAL_PLAN:
        cmd = [
            sys.executable,
            "-m",
            "modal",
            "run",
            "modal_train.py::train",
            "--agent",
            job["agent"],
            "--episodes",
            str(job["episodes"]),
            "--batch-size",
            str(job["batch_size"]),
        ]
        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)
else:
    print("Skipping optional runs. Set RUN_OPTIONAL = True when ready.")


## Evaluate

In [ ]:
import subprocess
import sys

for agent in ["holmes", "forge"]:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "modal",
            "run",
            "modal_train.py::evaluate",
            "--agent",
            agent,
            "--eval-episodes",
            "20",
        ],
        check=True,
    )


## Download Volume Artifacts

In [ ]:
import subprocess
import sys
from pathlib import Path

ARTIFACT_DIR = Path("/root/sentinel_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        sys.executable,
        "-m",
        "modal",
        "volume",
        "get",
        "sentinel-checkpoints",
        "/",
        str(ARTIFACT_DIR),
        "--force",
    ],
    check=True,
)
print(sorted(str(p.relative_to(ARTIFACT_DIR)) for p in ARTIFACT_DIR.rglob("*") if p.is_file())[:100])


## Plot Reward And Loss Curves

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ARTIFACT_DIR = Path("/root/sentinel_artifacts")
PLOT_DIR = ARTIFACT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)

for agent in ["holmes", "forge", "hermes", "oracle", "argus"]:
    log_path = ARTIFACT_DIR / f"{agent}_training_log.jsonl"
    if not log_path.exists():
        print(f"Skipping {agent}: no log found")
        continue

    df = load_jsonl(log_path)
    display(df.tail())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["episode"], df["total_reward"], label=f"{agent} total_reward")
    axes[0].set_title(f"{agent} reward curve")
    axes[0].set_xlabel("episode")
    axes[0].set_ylabel("total_reward")
    axes[0].legend()

    if "loss" in df.columns and df["loss"].notna().any():
        axes[1].plot(df["episode"], df["loss"], label=f"{agent} loss", color="orange")
    axes[1].set_title(f"{agent} loss curve")
    axes[1].set_xlabel("episode")
    axes[1].set_ylabel("loss")
    axes[1].legend()

    fig.tight_layout()
    out = PLOT_DIR / f"{agent}_training_curves.png"
    fig.savefig(out, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")


## Package Artifacts

In [ ]:
import shutil
from pathlib import Path

ARTIFACT_DIR = Path("/root/sentinel_artifacts")
zip_path = shutil.make_archive("/root/sentinel_judge_artifacts", "zip", ARTIFACT_DIR)
print(zip_path)
